# SV 73/2024 Danilo Torbica
## Interpolacione metode za obradu slike

Ovaj notebook dokumentuje deo projekta koji obuhvata:
- bilinearnu interpolaciju,
- bikubnu interpolaciju,
- interpolaciju kubnim splajnovima,
- rekonstrukciju nedostajućih piksela,
- povećanje rezolucije slike.

Fokus je na korišćenju postojećih implementacija iz foldera `src/interpolation`.

## 1. Priprema okruženja
Učitavanje funkcije iz projekta. Većina logike ostaje u posebnim Python fajlovima, a notebook služi za pregled eksperimenata i rezultata.

In [ ]:
import os
import sys
import time
import numpy as np
from PIL import Image
from IPython.display import display

cwd = os.getcwd()
if os.path.isdir(os.path.join(cwd, "src")):
    project_root = cwd
elif os.path.isdir(os.path.join(cwd, "..", "src")):
    project_root = os.path.abspath(os.path.join(cwd, ".."))
else:
    project_root = cwd

if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.image_io import load_image, save_image
from src.interpolation.bilinear import bilinear_interpolation
from src.interpolation.bicubic import bicubic_interpolation
from src.interpolation.spline import spline_interpolation

np.random.seed(42)
print("Projektni root:", project_root)
print("Import uspesan.")

## 2. Pomoćne funkcije
Definisanje pomoćnih funkcija za metriku i prikaz rezultata.

In [ ]:
def mse(img_a, img_b):
    diff = img_a.astype(np.float32) - img_b.astype(np.float32)
    return float(np.mean(diff ** 2))

def psnr(img_a, img_b, max_val=255.0):
    err = mse(img_a, img_b)
    if err == 0.0:
        return float("inf")
    return float(10.0 * np.log10((max_val ** 2) / err))

def show_image(np_img, title):
    print(title, "| shape:", np_img.shape)
    display(Image.fromarray(np.clip(np_img, 0, 255).astype(np.uint8)))

def maybe_plot_bar(labels, values, title, ylabel):
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 3.5))
        plt.bar(labels, values)
        plt.title(title)
        plt.ylabel(ylabel)
        plt.grid(axis="y", linestyle="--", alpha=0.35)
        plt.show()
    except Exception:
        print("Matplotlib nije dostupan - prikazujemo samo numericke vrednosti.")
        for lbl, val in zip(labels, values):
            print(f"  {lbl}: {val}")

## 3. Eksperiment: povećanje rezolucije (upscaling)
Korišćenje jedne test slike i upoređivanje vremena izvršavanja tri metode. Faktor skaliranja je `2x` radi bržeg testa.

In [ ]:
input_path = os.path.join("data", "input", "degraded", "lenna.png")
img = load_image(input_path)
show_image(img, "Ulazna slika")

methods = {
    "bilinear": bilinear_interpolation,
    "bicubic": bicubic_interpolation,
    "spline": spline_interpolation,
}

scale = 2
upscaled_results = {}
upscaled_times = {}

for name, fn in methods.items():
    t0 = time.perf_counter()
    out = fn(img, scale_factor=scale)
    dt = time.perf_counter() - t0

    upscaled_results[name] = out
    upscaled_times[name] = dt

    out_path = os.path.join("user", "output", f"notebook_output_{name}.png")
    save_image(out, out_path)
    print(f"{name:8s} | vreme: {dt:.4f}s | sacuvano: {out_path}")

show_image(upscaled_results["bilinear"], "Upscaled - bilinear")
show_image(upscaled_results["bicubic"], "Upscaled - bicubic")
show_image(upscaled_results["spline"], "Upscaled - spline")

labels = list(upscaled_times.keys())
values = [upscaled_times[k] for k in labels]
maybe_plot_bar(labels, values, "Vreme izvrsavanja - Upscaling", "sekunde")

Zaključak (upscaling):
- sve tri metode uspešno povećavaju rezoluciju,
- vreme izvršavanja i vizuelni detalji se razlikuju po metodi,
- rezultati su sačuvani u `user/output` za dalje poređenje.

## 4. Eksperiment: rekonstrukcija nedostajućih piksela
Pravimo veštačku masku nedostajućih piksela, rekonstruišemo sliku i računamo metrike `MSE` i `PSNR` u odnosu na original.

In [ ]:
original_path = os.path.join("data", "input", "original", "lenna.png")
degraded_path = os.path.join("data", "input", "degraded", "lenna.png")

img_original = load_image(original_path)
img_degraded = load_image(degraded_path)

h, w, _ = img_degraded.shape
missing_mask = np.random.rand(h, w) < 0.10
img_missing = img_degraded.copy()
img_missing[missing_mask] = 0

show_image(img_missing, "Slika sa nedostajucim pikselima (10%)")

recon_results = {}
recon_times = {}
recon_metrics = {}

for name, fn in methods.items():
    t0 = time.perf_counter()
    rec = fn(img_missing, missing_mask=missing_mask)
    dt = time.perf_counter() - t0

    recon_results[name] = rec
    recon_times[name] = dt
    recon_metrics[name] = {
        "MSE": mse(img_original, rec),
        "PSNR": psnr(img_original, rec),
    }

    out_path = os.path.join("user", "output_recon", f"notebook_recon_{name}.png")
    save_image(rec, out_path)

print("\nRezultati rekonstrukcije:")
for name in methods:
    print(
        f"{name:8s} | vreme: {recon_times[name]:.4f}s | "
        f"MSE: {recon_metrics[name]['MSE']:.2f} | PSNR: {recon_metrics[name]['PSNR']:.2f} dB"
    )

show_image(recon_results["bilinear"], "Rekonstrukcija - bilinear")
show_image(recon_results["bicubic"], "Rekonstrukcija - bicubic")
show_image(recon_results["spline"], "Rekonstrukcija - spline")

labels = list(methods.keys())
psnr_values = [recon_metrics[k]["PSNR"] for k in labels]
time_values = [recon_times[k] for k in labels]

maybe_plot_bar(labels, psnr_values, "PSNR - Rekonstrukcija", "dB")
maybe_plot_bar(labels, time_values, "Vreme - Rekonstrukcija", "sekunde")

Zaključak (rekonstrukcija):
- svaka metoda može da popuni nedostajuće piksele,
- kvalitet i brzina zavise od metode i od rasporeda maske,
- `PSNR` i `MSE` omogućavaju kvantitativno poređenje,
- rekonstruisane slike su sačuvane u `user/output_recon`.